# Model Serialization: Examples

This notebook demonstrates model persistence with `pickle` and `joblib`, a complete scikit-learn pipeline, metadata, and a basic post-load smoke test.

**Safety:** only deserialize artifacts you trust. Pickle/joblib files can execute code when loaded. Run the notebook from top to bottom. It uses a synthetic dataset and does not download data.

In [ ]:
from pathlib import Path
import json
import pickle
import platform
import hashlib
from datetime import datetime, timezone

import joblib
import sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

## 1. Create a reproducible synthetic dataset

In [ ]:
X_array, y = make_classification(
    n_samples=800,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    random_state=RANDOM_STATE,
)
feature_names = [f"feature_{i}" for i in range(1, X_array.shape[1] + 1)]
X = pd.DataFrame(X_array, columns=feature_names)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

## 2. Train and evaluate a pipeline

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
pipeline.fit(X_train, y_train)

original_predictions = pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, original_predictions)
print(f"Test accuracy: {test_accuracy:.3f}")
print(classification_report(y_test, original_predictions))

## 3. Serialize with pickle

In [ ]:
pickle_path = ARTIFACT_DIR / "classifier_pipeline.pkl"
with pickle_path.open("wb") as file:
    pickle.dump(pipeline, file)

# Only load pickle files from trusted sources.
with pickle_path.open("rb") as file:
    pickle_model = pickle.load(file)

pickle_predictions = pickle_model.predict(X_test)
assert np.array_equal(original_predictions, pickle_predictions)
print("Pickle round trip passed.")

## 4. Serialize with joblib

In [ ]:
joblib_path = ARTIFACT_DIR / "classifier_pipeline.joblib"
joblib.dump(pipeline, joblib_path)

# Only load joblib files from trusted sources.
joblib_model = joblib.load(joblib_path)
joblib_predictions = joblib_model.predict(X_test)
assert np.array_equal(original_predictions, joblib_predictions)
print("Joblib round trip passed.")
print("Artifact size (bytes):", joblib_path.stat().st_size)

## 5. Save and read metadata

In [ ]:
metadata = {
    "model_name": "serialization-demo-logistic-regression",
    "model_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
    "features": feature_names,
    "test_accuracy": float(test_accuracy),
    "artifact": joblib_path.name,
}
metadata_path = ARTIFACT_DIR / "metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

loaded_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print(json.dumps(loaded_metadata, indent=2))

## 6. Smoke test the loaded model

In [ ]:
def load_and_check(path, X_input, expected_features):
    # Trust boundary: only load files from trusted storage.
    model = joblib.load(path)
    actual_features = getattr(model, "feature_names_in_", None)

    if actual_features is not None and list(actual_features) != list(expected_features):
        raise ValueError(
            f"Feature mismatch: expected {list(expected_features)}, "
            f"got {list(actual_features)}"
        )
    if hasattr(X_input, "columns") and list(X_input.columns) != list(expected_features):
        raise ValueError("Input columns do not match expected feature names.")

    predictions = model.predict(X_input)
    if len(predictions) != len(X_input):
        raise RuntimeError("Unexpected prediction count.")
    return model, predictions

checked_model, checked_predictions = load_and_check(
    joblib_path, X_test, feature_names
)
assert np.array_equal(original_predictions, checked_predictions)
print("Smoke test passed; predictions match.")

## 7. Calculate a SHA-256 checksum

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("SHA-256:", sha256_file(joblib_path))

## 8. Key takeaways

- Save the preprocessing steps and estimator together in a pipeline.
- Store metadata separately in a readable format such as JSON.
- Record library versions and test inference after loading.
- Never load pickle/joblib artifacts from untrusted sources.
- A checksum detects byte changes only when compared against a trusted value; it does not establish that a file is safe.

### Cleanup (optional)

```python
# Uncomment to remove generated artifacts:
# for path in ARTIFACT_DIR.iterdir():
#     path.unlink()
# ARTIFACT_DIR.rmdir()
```